⏱️ **Time required:** ~1 minute | **Type:** Compliance scenario (run all cells)

# 🔐 GDPR Right-to-Be-Forgotten — Cross-Domain Erasure

**Owner:** Data Protection Officer / Legal  
**Regulation:** GDPR Art. 17, CCPA §1798.105

This notebook demonstrates how LakeLogic enforces privacy erasure **natively across the entire Data Mesh**. When a data subject requests deletion, we scan every domain's materialized Delta tables and surgically remove their PII — generating an audit-ready report.

> **Prerequisites:** Run `07_rideflow_marketplace.ipynb` first to populate the lakehouse.

---
## Step 1 · Setup & Registry

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb"  # 'polars' , 'spark'

In [ ]:
import os
import sys
from pathlib import Path
import polars as pl

# Point to the RideFlow reference architecture root
PROJECT_ROOT = Path(".").resolve()
LAKEHOUSE = PROJECT_ROOT / "lakehouse"

from lakelogic.core.registry import DomainRegistry

registry = DomainRegistry.from_yaml(
    str(PROJECT_ROOT / "assets" / "domains_rideflow" / "marketplace" / "rideflow" / "_system.yaml")
)

print(f"Domain  : {registry.domain}")
print(f"System  : {registry.system}")
print(f"Lakehouse: {LAKEHOUSE}")

---
## Step 2 · Identify the Data Subject

In production this would come from a ticketing system (Zendesk, ServiceNow). Here we pick a real rider from the Silver trips table.

In [ ]:
from lakelogic import read_delta  # polars' Delta bridge is broken; this routes via Arrow
# Read the Silver trips table to find a real rider_id
trips_path = LAKEHOUSE / registry.domain / registry.silver_layer / "silver_rideflow_trips"

if trips_path.exists():
    trips_df = read_delta(str(trips_path))
    sample_rider = trips_df.select("rider_id").unique().head(1).item()

    # Count their appearances across ALL tables
    rider_trips = trips_df.filter(pl.col("rider_id") == sample_rider)
    print(f"🎯 Data Subject: {sample_rider}")
    print(f"   Found in {len(rider_trips)} trip records")
else:
    sample_rider = "RDR-EXAMPLE"
    print(f"⚠️ No Silver trips found. Using placeholder: {sample_rider}")

TARGET_RIDER = sample_rider

---
## Step 3 · Pre-Erasure Audit (Before Snapshot)

Before executing any erasure, we capture the current state as evidence for the compliance audit trail.

In [ ]:
from lakelogic import read_delta  # polars' Delta bridge is broken; this routes via Arrow
print(f"📋 PRE-ERASURE AUDIT for {TARGET_RIDER}")
print("=" * 60)

domain_root = LAKEHOUSE / registry.domain
audit_before = {}

for layer in ["bronze", "silver", "gold"]:
    layer_path = domain_root / layer
    if not layer_path.exists():
        continue
    for table_dir in sorted(layer_path.iterdir()):
        if not table_dir.is_dir() or table_dir.name.startswith("_"):
            continue
        try:
            df = read_delta(str(table_dir))
            if "rider_id" in df.columns:
                matches = df.filter(pl.col("rider_id") == TARGET_RIDER)
                count = len(matches)
                audit_before[table_dir.name] = count
                status = f"🔴 {count} rows" if count > 0 else "✅ 0 rows"
                print(f"  {layer:7s} | {table_dir.name:45s} | {status}")
        except Exception as e:
            print(f"  {layer:7s} | {table_dir.name:45s} | ⚠️ {e}")

total_before = sum(audit_before.values())
print(f"\n  TOTAL PII EXPOSURE: {total_before} rows across {sum(1 for v in audit_before.values() if v > 0)} tables")

---
## Step 4 · Execute Right-to-Be-Forgotten

`run()` scans every contract-governed table and applies the chosen erasure
strategy. Passing `target_layers=""` selects no layers, so **only** the privacy
pass executes — nothing is reprocessed and no output is overwritten.

| Strategy | Effect | Use Case |
| :-- | :-- | :-- |
| `nullify` | Sets PII fields to NULL | Default — cleanest for analytics |
| `hash` | Replaces with salted SHA-256 | Preserves referential integrity |
| `redact` | Replaces with `[REDACTED]` | Visible audit trail |


In [ ]:
from lakelogic.pipeline.runner import LakehousePipeline

pipeline = LakehousePipeline(registry, engine=ENGINE)

ERASURE_STRATEGY = "nullify"  # Change to "hash" or "redact" to compare

print("🔐 Executing GDPR Erasure Pass")
print(f"   Subject  : {TARGET_RIDER}")
print("   Column   : rider_id")
print(f"   Strategy : {ERASURE_STRATEGY}")
print(f"   Scope    : All active contracts in {registry.domain}/{registry.system}")
print()

# target_layers="" runs the privacy pass ONLY -- no layer is reprocessed, so the
# erasure cannot be overwritten by a materialisation immediately afterwards.
pipeline.run(
    target_layers="",
    forget_column="rider_id",
    forget_values=[TARGET_RIDER],
    forget_strategy=ERASURE_STRATEGY,
    dry_run=False,
)

print()
print(f"✅ Erasure pass complete for {TARGET_RIDER}.")

### ⚙️ Execution Engine

---
## Step 5 · Post-Erasure Verification

Re-scan every table to confirm the subject's PII has been removed.

In [ ]:
from lakelogic import read_delta  # polars' Delta bridge is broken; this routes via Arrow
print(f"📋 POST-ERASURE VERIFICATION for {TARGET_RIDER}")
print("=" * 60)

audit_after = {}

for layer in ["bronze", "silver", "gold"]:
    layer_path = domain_root / layer
    if not layer_path.exists():
        continue
    for table_dir in sorted(layer_path.iterdir()):
        if not table_dir.is_dir() or table_dir.name.startswith("_"):
            continue
        try:
            df = read_delta(str(table_dir))
            if "rider_id" in df.columns:
                matches = df.filter(pl.col("rider_id") == TARGET_RIDER)
                count = len(matches)
                audit_after[table_dir.name] = count
                before = audit_before.get(table_dir.name, 0)
                if before > 0 and count == 0:
                    print(f"  {layer:7s} | {table_dir.name:45s} | ✅ ERASED ({before} → 0)")
                elif count > 0:
                    print(f"  {layer:7s} | {table_dir.name:45s} | ⚠️ STILL PRESENT: {count} rows")
                else:
                    print(f"  {layer:7s} | {table_dir.name:45s} | ✅ Clean (was 0)")
        except Exception as e:
            print(f"  {layer:7s} | {table_dir.name:45s} | ⚠️ {e}")

total_after = sum(audit_after.values())
print(f"\n  TOTAL REMAINING: {total_after} rows")
print(f"  ROWS ERASED: {total_before - total_after}")

---
## Step 6 · Erasure Audit Report

Generate a compliance-ready summary suitable for the DPO's records.

In [ ]:
from datetime import datetime, timezone

report = []
report.append("=" * 70)
report.append("GDPR ERASURE AUDIT REPORT")
report.append("=" * 70)
report.append(f"Date           : {datetime.now(timezone.utc).isoformat()}")
report.append(f"Data Subject   : {TARGET_RIDER}")
report.append("PII Column     : rider_id")
report.append(f"Strategy       : {ERASURE_STRATEGY}")
report.append(f"Domain         : {registry.domain}/{registry.system}")
report.append("")
report.append(f"Tables Scanned : {len(audit_before)}")
report.append(f"Tables Affected: {sum(1 for v in audit_before.values() if v > 0)}")
report.append(f"Rows Before    : {total_before}")
report.append(f"Rows After     : {total_after}")
report.append(f"Rows Erased    : {total_before - total_after}")
report.append("")
report.append(
    f"Verification   : {'✅ PASS — all PII removed' if total_after == 0 else '⚠️ FAIL — residual PII detected'}"
)
report.append("=" * 70)

for line in report:
    print(line)

---
## What You Just Saw

| Capability | Detail |
| :-- | :-- |
| **Subject Discovery** | Automatically located a real rider across all lakehouse tables |
| **Cross-Layer Erasure** | Swept Bronze, Silver, and Gold Delta tables in a single pass |
| **Strategy Flexibility** | Supports `nullify`, `hash` (salted SHA-256), and `redact` |
| **Audit Trail** | Before/after verification with compliance-ready report |
| **Contract-Driven** | Only scans tables governed by active Data Contracts |

### Try It Yourself

- Change `ERASURE_STRATEGY` to `"hash"` or `"redact"` and re-run to compare
- Add `driver_id` as a second erasure column for driver privacy requests
- Extend to scan the Payments domain (`assets/domains_rideflow/payments/stripe/_system.yaml`)